In [1]:
import numpy as np
import pandas as pd
import joblib
import os
import json

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [2]:
from pathlib import Path


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def save_json(path, payload):
    ensure_dir(Path(path).parent)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

In [9]:
def load_data():
    data = {
        "text": [

            # Positive
            "This product is amazing and works perfectly",
            "I love this item so much",
            "Excellent quality and fast delivery",
            "Very satisfied with this purchase",
            "Highly recommended product",
            "Customer service was fantastic",
            "Great value for the money",
            "I am extremely happy with this",
            "Works better than expected",
            "Five stars, will buy again",

            "Absolutely wonderful experience",
            "Superb quality and performance",
            "Very reliable and useful",
            "Loved it, great product",
            "Outstanding service",

            # Negative
            "Terrible experience, completely useless",
            "Worst purchase I have ever made",
            "Very disappointing product",
            "Not worth the money",
            "Poor quality and bad design",
            "The product broke after one day",
            "Extremely frustrating to use",
            "Waste of money and time",
            "Horrible customer service",
            "Do not recommend this",

            "Very bad quality",
            "Completely dissatisfied",
            "Stopped working immediately",
            "Cheap and unreliable",
            "Total waste"
        ],

        "label": [

            # Positive (1)
            1,1,1,1,1,1,1,1,1,1,
            1,1,1,1,1,

            # Negative (0)
            0,0,0,0,0,0,0,0,0,0,
            0,0,0,0,0
        ]
    }

    return pd.DataFrame(data)

In [10]:
df = load_data()

df

,text,label
0,This product is amazing and works perfectly,1
1,I love this item so much,1
2,Excellent quality and fast delivery,1
3,Very satisfied with this purchase,1
4,Highly recommended product,1
5,Customer service was fantastic,1
6,Great value for the money,1
7,I am extremely happy with this,1
8,Works better than expected,1
9,"Five stars, will buy again",1


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 24
Test size: 6


In [24]:


pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1,2),      # uni + bi-grams
        max_features=10000,
        min_df=1
    )),
    
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

In [25]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

In [26]:
preds = pipeline.predict(X_test)

f1 = f1_score(y_test, preds)

print("F1 Score:", f1)

F1 Score: 0.5


In [18]:
ensure_dir("models")

joblib.dump(
    pipeline,
    "models/sentiment_pipeline.joblib"
)

save_json(
    "models/metrics.json",
    {"f1_score": float(f1)}
)

print("Model & metrics saved.")

Model & metrics saved.


In [21]:
def predict_sentiment(text):
    model = joblib.load("models/sentiment_pipeline.joblib")
    
    pred = model.predict([text])[0]
    
    return "Positive" if pred == 1 else "Negative"


# Test examples
print(predict_sentiment("I absolutely love this product"))
print(predict_sentiment("This was a waste of money"))
print(predict_sentiment("Customer support was excellent"))
print(predict_sentiment("Very random quality but i like it"))

Positive
Negative
Positive
Negative


In [22]:
print("=== NLP PROJECT SUMMARY ===\n")

print("Model: TF-IDF + Logistic Regression")
print("Metric: F1 Score =", round(f1,3))

print("\nKey Insight:")
print("Simple linear models perform well when combined with strong text features.")

print("\nUse Case:")
print("Can be used for reviews, feedback, social media monitoring.")

=== NLP PROJECT SUMMARY ===

Model: TF-IDF + Logistic Regression
Metric: F1 Score = 0.5

Key Insight:
Simple linear models perform well when combined with strong text features.

Use Case:
Can be used for reviews, feedback, social media monitoring.
